# ARTI308 - Machine Learning Lab 5


### 1. Import Libraries


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

sns.set(style='whitegrid')

### 2. Load and Prepare Dataset


In [15]:
df = pd.read_csv('talabat_enhanced_orders.csv')

df['Order_Time'] = pd.to_datetime(df['Order_Time'])
df['Delivery_Time'] = pd.to_datetime(df['Delivery_Time'])

le = LabelEncoder()
categorical_cols = ['City', 'Payment_Method', 'Driver_Vehicle', 'Traffic_Level', 'Driver_Availability']
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print("Data loaded and preprocessed.")
print(df.head())

Data loaded and preprocessed.
   Order_ID User_ID  Restaurant_ID  Driver_ID      Item_Name  Quantity  \
0         1   U3522            358        485  Fried Chicken         3   
1         2   U9214            316         65       Sandwich         3   
2         3   U7307            357        309        Koshary         3   
3         4   U3612            420         32          Sushi         2   
4         5   U3492             73        364        Koshary         5   

   Total_Price          Order_Time       Delivery_Time  \
0       273.72 2025-06-16 08:32:00 2025-06-16 09:11:00   
1       365.82 2025-06-03 21:27:00 2025-06-03 22:00:00   
2       401.94 2025-06-01 14:48:00 2025-06-01 15:26:00   
3       221.18 2025-06-13 02:30:00 2025-06-13 03:22:00   
4       355.55 2025-06-06 09:48:00 2025-06-06 10:32:00   

   Delivery_Duration_Minutes  ...  Driver_Vehicle  Restaurant_Lat  \
0                         39  ...               2       31.195082   
1                         33  ...     

### Task 1: Create New Engineered Feature


In [18]:
df['Is_Weekend'] = df['Order_Time'].dt.dayofweek.isin([4, 5]).astype(int) 

print("Feature 'Is_Weekend' created successfully.")

Feature 'Is_Weekend' created successfully.


### Task 2: Peak Hour Logic

In [21]:
df['is_peak_hour'] = df['Order_Time'].dt.hour.isin([12, 13, 19, 20]).astype(int)

print("Peak hour rule applied.")

Peak hour rule applied.


### Task 3: Category Reduction (Item_Name)


In [22]:
top_k = 30
top_items = df['Item_Name'].value_counts().nlargest(top_k).index
df['Item_Name_reduced'] = df['Item_Name'].apply(lambda x: x if x in top_items else 'Other')

df['Item_Name_reduced'] = le.fit_transform(df['Item_Name_reduced'])
print(f"Top {top_k} items kept, others grouped as 'Other'.")

Top 30 items kept, others grouped as 'Other'.


### Task 4: Feature Selection & Baseline Model


In [24]:
features = ['Quantity', 'Total_Price', 'Delivery_Duration_Minutes', 'City', 
            'Payment_Method', 'Is_Weekend', 'is_peak_hour', 'Item_Name_reduced', 'Delivery_Distance_km']
X = df[features]
y = df['Order_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Baseline Model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns)
print("Feature Importances:\n", importances.sort_values(ascending=False))

# Select top features
selected_features = importances.sort_values(ascending=False).head(5).index
X_selected = X[selected_features]

# Retrain and evaluate
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_selected, y, test_size=0.2, random_state=42)
rf_s = RandomForestClassifier(random_state=42)
rf_s.fit(X_train_s, y_train_s)

print(f"\nAccuracy After Feature Selection: {accuracy_score(y_test_s, rf_s.predict(X_test_s)):.4f}")

Feature Importances:
 Delivery_Distance_km         0.280518
Total_Price                  0.279244
Delivery_Duration_Minutes    0.161087
Item_Name_reduced            0.086891
City                         0.072984
Quantity                     0.050337
Payment_Method               0.036156
Is_Weekend                   0.016747
is_peak_hour                 0.016035
dtype: float64

Accuracy After Feature Selection: 0.8447


### Final evaluation report

In [25]:
y_pred = rf_s.predict(X_test_s)
print("\nClassification Report:\n", classification_report(y_test_s, y_pred))


Classification Report:
               precision    recall  f1-score   support

   Cancelled       0.11      0.00      0.00      2005
   Delivered       0.85      1.00      0.92     16922
  In Transit       0.00      0.00      0.00      1073

    accuracy                           0.84     20000
   macro avg       0.32      0.33      0.31     20000
weighted avg       0.73      0.84      0.78     20000

